In [ ]:
%pylab inline

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [ ]:
import pandas as pd
from glob import glob
from tqdm import tqdm
from skimage import io

In [ ]:
from skimage.transform import rotate, resize
from skimage.exposure import equalize_adapthist

In [ ]:
path_images = './00.data/cropped_old/'
path_labels = './RotatedImageFiles.txt'

In [ ]:
data_files = glob(path_images+'*')
data_files_zero = [path for path in data_files if '_0.jpg' in path]
data_labels = pd.read_csv(path_labels)

In [ ]:
data_labels.index = data_labels.Filename.values

In [ ]:
def get_image(img_path):
    
    img = io.imread(img_path)
    
    return(img)


def get_label(img_path,out=data_labels):
    
    img_id = img_path.split('/')[-1]
    row = out.loc[img_id]
    angle = row['Doppler_Angle']
    
    return(angle)

In [ ]:
data_x = [get_image(path) for path in data_files]
data_y = [get_label(path) for path in data_files]

In [ ]:
data_x_zero = [get_image(path) for path in data_files_zero]
data_y_zero = [get_label(path) for path in data_files_zero]

In [ ]:
angle_set = np.arange(-30,35,5)
rotation_angle = -25#np.random.choice(a = angle_set)
image_index = np.random.choice(np.arange(0,len(data_x_zero)))

img_x = data_x_zero[image_index]
img_y = np.round(data_y_zero[image_index],2)
new_angle = np.round(img_y+rotation_angle,2)


img_x = resize(img_x,(110,110),mode='reflect')
img_rot = rotate(img_x,angle=rotation_angle,mode='reflect')


print(img_y,rotation_angle,new_angle)

plt.figure(figsize=(10,6))

plt.subplot(131)
plt.imshow(img_x,)
plt.title('Original : '+str(img_y));

plt.subplot(132)
plt.imshow(img_rot)
plt.title('New : Original'+'+'+str(rotation_angle)+' = '+str(new_angle));

plt.subplot(133)
plt.imshow(equalize_adapthist(img_rot));
plt.title('Normalized : '+str(new_angle));

In [ ]:
img_path = data_files_zero[0]
img_id = img_path.split('/')[-1]
img_zero = plt.imread(img_path)

img_cols = 10
rotation_step = 2

i = 1

print(data_labels.loc[img_id]['Doppler_Angle'])
plt.figure(figsize=(20,4))
for ix in range(img_cols):
    angle_pos = ix*rotation_step
    angle_neg = -1*ix*rotation_step
    
    img_rot01 = rotate(img_zero,angle=angle_pos,mode='reflect')
    img_rot02 = rotate(img_zero,angle=angle_neg,mode='reflect')
    
    plt.subplot(2,img_cols,i)
    plt.imshow(img_rot01)
    plt.title(angle_pos)
    plt.axis('off')
    
    plt.subplot(2,img_cols,i+img_cols)
    plt.imshow(img_rot02)
    plt.axis('off')
    
    plt.title(angle_neg);
    
    i+=1

In [ ]:
def image_generator(input_data,
                    output_data,
                    model_pretrained,
                    batch_size=25,
                    img_channels=3,
                    rotation_range=60,
                    rotation_step=5,
                    negative=False,
                    input_size=(100, 100),
                    equalize_hist=True):

    img_rows, img_cols = input_size

    while True:
        
        batch_files = np.random.choice(input_data, batch_size, replace=True)
        
        if negative:
            start_range = -1 * rotation_range
        else:
            start_range = 0
            
        angle_set = np.arange(start_range,rotation_range,rotation_step)
        
        batch_images = []
        batch_labels = []

        for current_file in batch_files:

            rotation_angle = np.random.choice(a = angle_set)
            current_image = get_image(img_path = current_file)
            
            current_label = get_label(img_path=current_file, out=output_data)
            new_angle = current_label+rotation_angle
            
            
            if equalize_hist:
                current_image = equalize_adapthist(image=current_image)
                
            if new_angle>180:
                new_angle-=180

            current_image = resize(image=current_image, output_shape=(img_rows,img_cols), mode='reflect')
            current_image = rotate(image=current_image, angle=rotation_angle, mode='reflect')
            current_image = np.stack([current_image, current_image, current_image], axis=-1)
            
            batch_images.append(current_image)
            batch_labels.append(new_angle)

        batch_images = np.array(batch_images)
        batch_images = model_pretrained.predict(batch_images)
        batch_labels = np.array(batch_labels)
        
        #batch_images = np.expand_dims(batch_images,-1)
        batch_labels = np.expand_dims(np.expand_dims(np.expand_dims(batch_labels,1),1),1)
        
        batch_x, batch_y = batch_images, batch_labels

        yield (batch_x, batch_y)

In [ ]:
from keras import backend as K
from keras.regularizers import l2
from keras.optimizers import Adam
from keras.models import Model, load_model
from keras.callbacks import ModelCheckpoint,EarlyStopping
from keras.layers import Input, BatchNormalization, Conv2D, Dense, Activation, Dropout

In [ ]:
from keras.applications import Xception,VGG16,VGG19,ResNet50
from keras.applications import InceptionV3, InceptionResNetV2
from keras.applications import DenseNet121, DenseNet169, DenseNet201

variants = ['Xception', 'VGG16', 'VGG19', 
            'ResNet50', 'InceptionV3', 'InceptionResNetV2', 
            'DenseNet121', 'DenseNet169', 'DenseNet201']

In [ ]:
angle_range = 30
rotation_step = 2

train_generator = image_generator(input_data=data_files_zero[:-10],
                                  output_data=data_labels,
                                  model_pretrained=model_pretrained,
                                  input_size=img_size[:2],
                                  rotation_range=angle_range,
                                  rotation_step=rotation_step,
                                  negative=True)
test_generator = image_generator(input_data=data_files_zero[-10:],
                                 output_data=data_labels,
                                 model_pretrained=model_pretrained,
                                 input_size=img_size[:2],
                                 rotation_range=angle_range,
                                  rotation_step=rotation_step,
                                 negative=True)

In [ ]:
retrain = False

In [ ]:
def create_model(variant, dropout_rate=0.25):
    
    K.clear_session()

    if variant =='Xception':
        img_size = (139,139,3)
        model_pretrained = Xception(include_top=False,input_shape=img_size)

    elif variant =='VGG16':
        img_size = (139,139,3)
        model_pretrained = VGG16(include_top=False,input_shape=img_size)

    elif variant =='VGG19':
        img_size = (139,139,3)
        model_pretrained = VGG19(include_top=False,input_shape=img_size)

    elif variant =='ResNet50':
        img_size = (197,197,3)
        model_pretrained = ResNet50(include_top=False,input_shape=img_size)

    elif variant =='InceptionV3':
        img_size = (139,139,3)
        model_pretrained = InceptionV3(include_top=False,input_shape=img_size)

    elif variant =='InceptionResNetV2':
        img_size = (139,139,3)
        model_pretrained = InceptionResNetV2(include_top=False,input_shape=img_size)

    elif variant =='DenseNet121':
        img_size = (221,221,3)
        model_pretrained = DenseNet121(include_top=False,input_shape=img_size)

    elif variant =='DenseNet169':
        img_size = (221,221,3)
        model_pretrained = DenseNet169(include_top=False,input_shape=img_size)

    elif variant =='DenseNet201':
        img_size = (221,221,3)
        model_pretrained = DenseNet201(include_top=False,input_shape=img_size)

    elif variant =='NASNetLarge':
        img_size = (331,331,3)
        model_pretrained = NASNetLarge(include_top=False,input_shape=img_size)
    
    return(model_pretrained)

In [ ]:
for variant in variants:
    
    model_pretrained = create_model(variant)
    model_pretrained._make_predict_function()
    
    feature_shape = model_pretrained.output_shape
    n = feature_shape[1]
    
    features_shape = model_pretrained.output.shape.as_list()[1:]
    main_input = Input(shape=features_shape)

    x = main_input

    for i in range(n/2):
        
        r = x.shape.as_list()[1]
        if r>1:
            x = BatchNormalization()(x)
            x = Conv2D(filters=16*(i+1), 
                       kernel_size=(2,2), 
                       strides=(2,2),
                       kernel_regularizer=l2(l=1))(x)
            x = Dropout(dropout_rate)(x)

    x = BatchNormalization()(x)
    x = Dense(units=128, kernel_regularizer=l2(l=1))(x)
    x = Dropout(dropout_rate)(x)

    x = Dense(units=1,kernel_regularizer=l2(l=1))(x)
    main_output = Activation('relu')(x)

    model = Model(inputs=[main_input], outputs=[main_output])
    
    print(variant,model.input_shape,model.output_shape)
    model.summary()

In [ ]:
run_version = 'model_'+variant+'_'+str(angle_range)

MC = ModelCheckpoint(filepath='./02.model/'+run_version+'.h5')
ES = EarlyStopping(patience=20)

model.compile(optimizer=Adam(1e-3),loss='mse')

if retrain:
    model = load_model('./02.model/'+run_version+'.h5')

model.fit_generator(generator=train_generator,
                    validation_data = test_generator,
                    callbacks=[MC,ES],
                    steps_per_epoch=100,
                    validation_steps=10,
                    epochs=100,
                    verbose=2)